# Validate Market Data

This notebook validates the downloaded stock price data and NASDAQ index data before return calculation and event-study analysis.

It performs the following checks:

- verifies that the raw stock price file and NASDAQ index file exist
- verifies that required columns are present
- parses and validates date fields
- checks for duplicate rows
- checks for missing values in key columns
- verifies ticker coverage against the event metadata
- verifies that each event trading day has matching stock and index data
- summarizes the date coverage of the downloaded data

This notebook does **not** modify the raw datasets. It only verifies that the market data download pipeline output is correct.

In [1]:
import pandas as pd
from pathlib import Path

## 1. Define file paths

We first define the file paths used in the validation process.

In [2]:
metadata_path = Path("../data/processed/event_metadata_final.csv")
prices_path = Path("../data/raw/market_data/prices_raw.csv")
index_path = Path("../data/raw/market_data/nasdaq_index_raw.csv")

## 2. Check that required files exist

This step verifies that the metadata file, stock price file, and NASDAQ index file are present before loading them.

In [3]:
required_files = {
    "event metadata": metadata_path,
    "stock prices": prices_path,
    "NASDAQ index": index_path,
}

for name, path in required_files.items():
    print(f"{name}: {path.exists()} -> {path}")
    assert path.exists(), f"Missing required file: {path}"

event metadata: True -> ../data/processed/event_metadata_final.csv
stock prices: True -> ../data/raw/market_data/prices_raw.csv
NASDAQ index: True -> ../data/raw/market_data/nasdaq_index_raw.csv


## 3. Load datasets

We now load the event metadata, stock price data, and NASDAQ index data into pandas DataFrames.

In [4]:
metadata = pd.read_csv(metadata_path)
prices = pd.read_csv(prices_path)
index_df = pd.read_csv(index_path)

print("metadata shape:", metadata.shape)
print("prices shape:", prices.shape)
print("index shape:", index_df.shape)

metadata shape: (188, 15)
prices shape: (13170, 8)
index shape: (1317, 7)


## 4. Preview the datasets

A quick preview helps confirm that the files were loaded correctly and have the expected structure.

In [5]:
display(metadata.head())
display(prices.head())
display(index_df.head())

,ticker,file_name,file_path,call_date_from_filename,year,quarter_label,call_datetime_header_raw,call_datetime_parsed,call_datetime_gmt,call_datetime_et,call_time_gmt,call_time_et,after_market_close_et,event_trading_day_final,error
0,AAPL,2016-Jan-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-01-26,2016,Q1,"JANUARY 26, 2016 / 10:00PM GMT",2016-01-26T22:00:00+00:00,2016-01-26T22:00:00+00:00,2016-01-26T17:00:00-05:00,22:00:00,17:00:00,True,2016-01-27,NaN
1,AAPL,2016-Apr-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-04-26,2016,Q2,"APRIL 26, 2016 / 9:00PM GMT",2016-04-26T21:00:00+00:00,2016-04-26T21:00:00+00:00,2016-04-26T17:00:00-04:00,21:00:00,17:00:00,True,2016-04-27,NaN
2,AAPL,2016-Jul-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-07-26,2016,Q3,"JULY 26, 2016 / 9:00PM GMT",2016-07-26T21:00:00+00:00,2016-07-26T21:00:00+00:00,2016-07-26T17:00:00-04:00,21:00:00,17:00:00,True,2016-07-27,NaN
3,AAPL,2016-Oct-25-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-10-25,2016,Q4,"OCTOBER 25, 2016 / 9:00PM GMT",2016-10-25T21:00:00+00:00,2016-10-25T21:00:00+00:00,2016-10-25T17:00:00-04:00,21:00:00,17:00:00,True,2016-10-26,NaN
4,AAPL,2017-Jan-31-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2017-01-31,2017,Q1,"JANUARY 31, 2017 / 10:00PM GMT",2017-01-31T22:00:00+00:00,2017-01-31T22:00:00+00:00,2017-01-31T17:00:00-05:00,22:00:00,17:00:00,True,2017-02-01,NaN


,Date,ticker,Open,High,Low,Close,Adj Close,Volume
0,2015-06-29,AAPL,31.365000,31.617500,31.120001,31.132500,27.805973,196645600
1,2015-06-30,AAPL,31.392500,31.530001,31.215000,31.357500,28.006931,177482800
2,2015-07-01,AAPL,31.725000,31.735001,31.497499,31.650000,28.268177,120955200
3,2015-07-02,AAPL,31.607500,31.672501,31.442499,31.610001,28.232452,108844000
4,2015-07-06,AAPL,31.235001,31.557501,31.212500,31.500000,28.134207,112241600


,Date,Adj Close,Close,High,Low,Open,Volume
0,2015-06-29,4958.470215,4958.470215,5051.009766,4956.229980,5021.209961,2025580000
1,2015-06-30,4986.870117,4986.870117,5008.759766,4968.259766,5000.149902,2034430000
2,2015-07-01,5013.120117,5013.120117,5038.549805,4994.459961,5029.049805,1814560000
3,2015-07-02,5009.209961,5009.209961,5027.470215,4990.740234,5024.299805,1490810000
4,2015-07-06,4991.939941,4991.939941,5020.709961,4960.930176,4963.799805,1741500000


## 5. Validate required columns

This step checks that each dataset contains the required columns needed for the event study pipeline.

In [6]:
required_metadata_cols = ["ticker", "event_trading_day_final"]
required_prices_cols = ["Date", "ticker", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
required_index_cols = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]

missing_metadata_cols = [c for c in required_metadata_cols if c not in metadata.columns]
missing_prices_cols = [c for c in required_prices_cols if c not in prices.columns]
missing_index_cols = [c for c in required_index_cols if c not in index_df.columns]

print("Missing metadata columns:", missing_metadata_cols)
print("Missing prices columns:", missing_prices_cols)
print("Missing index columns:", missing_index_cols)

assert not missing_metadata_cols, f"Metadata missing columns: {missing_metadata_cols}"
assert not missing_prices_cols, f"Prices missing columns: {missing_prices_cols}"
assert not missing_index_cols, f"Index missing columns: {missing_index_cols}"

Missing metadata columns: []
Missing prices columns: []
Missing index columns: []


## 6. Parse date columns

We convert the event trading day and market date fields into pandas datetime format and check whether any values fail to parse.

In [7]:
metadata["event_trading_day_final"] = pd.to_datetime(metadata["event_trading_day_final"], errors="coerce")
prices["Date"] = pd.to_datetime(prices["Date"], errors="coerce")
index_df["Date"] = pd.to_datetime(index_df["Date"], errors="coerce")

print("Unparsed metadata event dates:", metadata["event_trading_day_final"].isna().sum())
print("Unparsed stock price dates:", prices["Date"].isna().sum())
print("Unparsed index dates:", index_df["Date"].isna().sum())

assert metadata["event_trading_day_final"].notna().all(), "Some event_trading_day_final values could not be parsed"
assert prices["Date"].notna().all(), "Some stock price dates could not be parsed"
assert index_df["Date"].notna().all(), "Some index dates could not be parsed"

Unparsed metadata event dates: 0
Unparsed stock price dates: 0
Unparsed index dates: 0


## 7. Standardize ordering

We sort the stock price and index datasets by date so that later return calculations are based on correctly ordered observations.

In [8]:
prices = prices.sort_values(["ticker", "Date"]).reset_index(drop=True)
index_df = index_df.sort_values("Date").reset_index(drop=True)

## 8. Check for duplicate rows

We verify that stock prices do not contain duplicate `(ticker, Date)` combinations and that the index file does not contain duplicate `Date` rows.

In [9]:
duplicate_price_rows = prices.duplicated(subset=["ticker", "Date"]).sum()
duplicate_index_rows = index_df.duplicated(subset=["Date"]).sum()

print("Duplicate stock price rows (ticker, Date):", duplicate_price_rows)
print("Duplicate index rows (Date):", duplicate_index_rows)

assert duplicate_price_rows == 0, "Duplicate (ticker, Date) rows found in stock prices"
assert duplicate_index_rows == 0, "Duplicate Date rows found in NASDAQ index data"

Duplicate stock price rows (ticker, Date): 0
Duplicate index rows (Date): 0


## 9. Check for missing values in key columns

This step verifies that essential variables needed for return calculation are populated.

In [10]:
price_key_cols = ["Date", "ticker", "Close", "Adj Close", "Volume"]
index_key_cols = ["Date", "Close", "Adj Close", "Volume"]

print("Missing values in stock price key columns:")
print(prices[price_key_cols].isna().sum())

print("\nMissing values in index key columns:")
print(index_df[index_key_cols].isna().sum())

Missing values in stock price key columns:
Date         0
ticker       0
Close        0
Adj Close    0
Volume       0
dtype: int64

Missing values in index key columns:
Date         0
Close        0
Adj Close    0
Volume       0
dtype: int64


## 10. Check ticker coverage against event metadata

We compare the tickers present in the event metadata to the tickers present in the stock price dataset.

In [11]:
metadata_tickers = set(metadata["ticker"].dropna().astype(str).str.strip().unique())
price_tickers = set(prices["ticker"].dropna().astype(str).str.strip().unique())

missing_in_prices = sorted(metadata_tickers - price_tickers)
extra_in_prices = sorted(price_tickers - metadata_tickers)

print("Tickers in metadata:", len(metadata_tickers))
print("Tickers in prices:", len(price_tickers))
print("Missing in prices:", missing_in_prices)
print("Extra in prices:", extra_in_prices)

assert len(missing_in_prices) == 0, f"Some metadata tickers are missing in prices: {missing_in_prices}"

Tickers in metadata: 10
Tickers in prices: 10
Missing in prices: []
Extra in prices: []


## 11. Summarize date coverage

We inspect the overall date range of the stock price and index data to confirm that the download window covers the event-study requirements.

In [12]:
price_min_date = prices["Date"].min()
price_max_date = prices["Date"].max()

index_min_date = index_df["Date"].min()
index_max_date = index_df["Date"].max()

event_min_date = metadata["event_trading_day_final"].min()
event_max_date = metadata["event_trading_day_final"].max()

print("Event date range :", event_min_date.date(), "to", event_max_date.date())
print("Stock date range :", price_min_date.date(), "to", price_max_date.date())
print("Index date range :", index_min_date.date(), "to", index_max_date.date())

assert price_min_date <= event_min_date, "Stock price data starts after earliest event date"
assert price_max_date >= event_max_date, "Stock price data ends before latest event date"
assert index_min_date <= event_min_date, "Index data starts after earliest event date"
assert index_max_date >= event_max_date, "Index data ends before latest event date"

Event date range : 2016-01-15 to 2020-08-20
Stock date range : 2015-06-29 to 2020-09-18
Index date range : 2015-06-29 to 2020-09-18


## 12. Verify event trading day coverage in stock data

For each event, we check whether the corresponding ticker has stock price data on its event trading day.

In [13]:
price_keys = set(
    zip(
        prices["ticker"].astype(str),
        prices["Date"].dt.normalize()
    )
)

event_check = metadata[["ticker", "event_trading_day_final"]].copy()
event_check["event_trading_day_final"] = event_check["event_trading_day_final"].dt.normalize()

event_check["stock_price_available"] = event_check.apply(
    lambda row: (str(row["ticker"]), row["event_trading_day_final"]) in price_keys,
    axis=1
)

missing_stock_event_days = event_check.loc[~event_check["stock_price_available"]].copy()

print("Events checked:", len(event_check))
print("Events missing stock price on event trading day:", len(missing_stock_event_days))

display(missing_stock_event_days.head(20))

Events checked: 188
Events missing stock price on event trading day: 0


,ticker,event_trading_day_final,stock_price_available


## 13. Verify event trading day coverage in index data

For each event, we check whether the NASDAQ index file contains data for the event trading day.

In [14]:
index_dates = set(index_df["Date"].dt.normalize())

event_check["index_available"] = event_check["event_trading_day_final"].apply(
    lambda d: d in index_dates
)

missing_index_event_days = event_check.loc[~event_check["index_available"]].copy()

print("Events missing index data on event trading day:", len(missing_index_event_days))

display(missing_index_event_days.head(20))

Events missing index data on event trading day: 0


,ticker,event_trading_day_final,stock_price_available,index_available


## 14. Investigate missing event-day matches

If any event trading days are missing in the stock or index data, we inspect them here. In a correctly downloaded dataset, these counts should usually be zero.

In [15]:
if len(missing_stock_event_days) == 0 and len(missing_index_event_days) == 0:
    print("All event trading days are covered by both stock price data and index data.")
else:
    print("Missing stock event days:")
    display(missing_stock_event_days)

    print("Missing index event days:")
    display(missing_index_event_days)

All event trading days are covered by both stock price data and index data.


## 15. Check observation counts by ticker

This step helps detect unexpectedly sparse ticker histories that could later affect estimation-window construction.

In [16]:
ticker_counts = (
    prices.groupby("ticker")
    .size()
    .reset_index(name="n_rows")
    .sort_values("n_rows")
)

display(ticker_counts)

,ticker,n_rows
0,AAPL,1317
1,AMD,1317
2,AMZN,1317
3,ASML,1317
4,CSCO,1317
5,GOOGL,1317
6,INTC,1317
7,MSFT,1317
8,MU,1317
9,NVDA,1317


## 16. Basic numeric sanity checks

We verify that core price fields are non-negative and inspect whether any clearly invalid values appear in the raw data.

In [17]:
print("Stock data numeric checks")
print("Negative Close values:", (prices["Close"] < 0).sum())
print("Negative Adj Close values:", (prices["Adj Close"] < 0).sum())
print("Negative Volume values:", (prices["Volume"] < 0).sum())

print("\nIndex data numeric checks")
print("Negative Close values:", (index_df["Close"] < 0).sum())
print("Negative Adj Close values:", (index_df["Adj Close"] < 0).sum())
print("Negative Volume values:", (index_df["Volume"] < 0).sum())

assert (prices["Close"] >= 0).all(), "Negative Close values found in stock data"
assert (prices["Adj Close"] >= 0).all(), "Negative Adj Close values found in stock data"
assert (prices["Volume"] >= 0).all(), "Negative Volume values found in stock data"

assert (index_df["Close"] >= 0).all(), "Negative Close values found in index data"
assert (index_df["Adj Close"] >= 0).all(), "Negative Adj Close values found in index data"
assert (index_df["Volume"] >= 0).all(), "Negative Volume values found in index data"

Stock data numeric checks
Negative Close values: 0
Negative Adj Close values: 0
Negative Volume values: 0

Index data numeric checks
Negative Close values: 0
Negative Adj Close values: 0
Negative Volume values: 0


## 17. Validation summary

This final section summarizes whether the downloaded stock price and NASDAQ index data passed the main structural checks needed before return calculation.

In [18]:
summary = {
    "metadata_rows": len(metadata),
    "stock_price_rows": len(prices),
    "index_rows": len(index_df),
    "metadata_tickers": len(metadata_tickers),
    "price_tickers": len(price_tickers),
    "missing_metadata_tickers_in_prices": len(missing_in_prices),
    "duplicate_stock_rows": duplicate_price_rows,
    "duplicate_index_rows": duplicate_index_rows,
    "missing_stock_event_days": len(missing_stock_event_days),
    "missing_index_event_days": len(missing_index_event_days),
}

summary_df = pd.DataFrame(summary.items(), columns=["check", "value"])
display(summary_df)

,check,value
0,metadata_rows,188
1,stock_price_rows,13170
2,index_rows,1317
3,metadata_tickers,10
4,price_tickers,10
5,missing_metadata_tickers_in_prices,0
6,duplicate_stock_rows,0
7,duplicate_index_rows,0
8,missing_stock_event_days,0
9,missing_index_event_days,0


## Reflection: Market Data Validation
The market data validation process confirms that the data pipeline for stock prices and NASDAQ index data is structurally sound and fully consistent with the event-study design.

The dataset contains 188 earnings call events across 10 unique tickers, with complete price coverage for all tickers. The downloaded stock price dataset includes 13,170 observations, while the NASDAQ index dataset contains 1,317 observations, indicating sufficient historical depth to support estimation and event windows.

All structural validation checks passed:

No missing tickers between the event metadata and price dataset

No duplicate observations in either stock or index data

No missing event trading days in stock prices

No missing event trading days in index data

This confirms that the event alignment logic based on event_trading_day_final is correctly implemented, and that market data has been successfully synchronized with the earnings call events.

The absence of missing event-day matches is particularly important, as it ensures that all events can be included in subsequent return calculations and CAR estimation without requiring imputation or exclusion.

Additionally, the date coverage of the dataset fully spans the required windows for:

estimation ([-120, -20])

event windows ([0,1], [0,3])

volatility analysis ([-10,-1], [+1,+10])

This indicates that the download window logic (−200 to +30 calendar days) is appropriate and sufficient.

Overall, the validation results provide strong evidence that the market data pipeline is:

complete

internally consistent

aligned with the methodological design

ready for return calculation and event-study analysis